# Manager portfolio-similarity audit

This notebook audits the quarterly manager portfolio-similarity networks produced by `scripts/09_build_manager_similarity.py`.

It checks:

1. the integrity of the saved 25-nearest-neighbor edge table;
2. agreement between edge endpoints and ownership-network manager nodes;
3. reconciliation of the baseline edge table with its quarterly audit; and
4. sensitivity to using 10, 25, or 50 neighbors.

The notebook reads saved outputs only and does not rebuild portfolio similarities.

## 1. Setup and inputs

In [1]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.parquet as pq


project_root = Path.cwd().resolve()

while not (project_root / "config" / "paths.yaml").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not locate the project root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.configuration import get_paths_config


paths = get_paths_config()
network_directory = paths.interim / "networks"
manager_path = network_directory / "ownership" / "manager_nodes.parquet"
similarity_directory = network_directory / "manager_similarity"
edges_path = similarity_directory / "manager_similarity_edges.parquet"
audit_path = similarity_directory / "similarity_graph_audit.parquet"

table_paths = {
    "manager_nodes": manager_path,
    "manager_similarity_edges": edges_path,
    "similarity_graph_audit": audit_path,
}

for name, path in table_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"Required {name} file not found: {path}")

inventory = pd.DataFrame(
    {
        "table": table_paths.keys(),
        "rows": [
            pq.ParquetFile(path).metadata.num_rows
            for path in table_paths.values()
        ],
    }
)

connection = duckdb.connect(database=":memory:")
connection.read_parquet(str(manager_path)).create_view("manager_nodes")
connection.read_parquet(str(edges_path)).create_view("similarity_edges")
similarity_audit = pd.read_parquet(audit_path)

display(inventory.style.format({"rows": "{:,}"}))

,table,rows
0,manager_nodes,"251,386"
1,manager_similarity_edges,"5,627,437"
2,similarity_graph_audit,156


## 2. Baseline edge integrity

The production edge table is the undirected union of each manager's 25 nearest positive-similarity neighbors. Every pair is stored once with the lower manager index first.

In [2]:
edge_checks = connection.execute(
    '''
    WITH joined AS (
        SELECT
            e.*,
            a.CIK AS node_a_cik,
            b.CIK AS node_b_cik,
            a.information_date AS node_a_date,
            b.information_date AS node_b_date
        FROM similarity_edges AS e
        LEFT JOIN manager_nodes AS a
            ON e.report_period = a.report_period
           AND e.manager_a_index = a.manager_index
        LEFT JOIN manager_nodes AS b
            ON e.report_period = b.report_period
           AND e.manager_b_index = b.manager_index
    )
    SELECT
        COUNT(*) - COUNT(DISTINCT (
            report_period, manager_a_index, manager_b_index
        )) AS duplicate_edge_keys,
        COUNT(*) FILTER (WHERE manager_a_index >= manager_b_index)
            AS unordered_or_self_edges,
        COUNT(*) FILTER (
            WHERE node_a_cik IS NULL OR node_b_cik IS NULL
        ) AS missing_manager_endpoints,
        COUNT(*) FILTER (
            WHERE manager_a_cik != node_a_cik
               OR manager_b_cik != node_b_cik
        ) AS endpoint_cik_mismatches,
        COUNT(*) FILTER (
            WHERE information_date != node_a_date
               OR information_date != node_b_date
        ) AS information_date_mismatches,
        COUNT(*) FILTER (
            WHERE cosine_similarity <= 0 OR cosine_similarity > 1
               OR NOT isfinite(cosine_similarity)
        ) AS invalid_similarities,
        COUNT(*) FILTER (
            WHERE directional_links NOT IN (1, 2)
               OR is_mutual_neighbor != (directional_links = 2)
        ) AS invalid_directional_links,
        COUNT(*) FILTER (
            WHERE best_neighbor_rank NOT BETWEEN 1 AND 25
               OR k_neighbors != 25
        ) AS invalid_neighbor_parameters
    FROM joined
    '''
).fetchdf().T.reset_index()
edge_checks.columns = ["check", "violations"]

display(edge_checks.style.format({"violations": "{:,}"}))
print()
print(f"Total baseline-edge violations: {edge_checks['violations'].sum():,}")

,check,violations
0,duplicate_edge_keys,0
1,unordered_or_self_edges,0
2,missing_manager_endpoints,0
3,endpoint_cik_mismatches,0
4,information_date_mismatches,0
5,invalid_similarities,0
6,invalid_directional_links,0
7,invalid_neighbor_parameters,0



Total baseline-edge violations: 0


## 3. Baseline audit reconciliation

The saved 25-neighbor audit should reproduce the number of edges, mutual links, connected managers, similarity statistics, and projected density in the production edge table.

In [3]:
connection.register("saved_audit", similarity_audit)
baseline_reconciliation = connection.execute(
    '''
    WITH endpoints AS (
        SELECT report_period, manager_a_index AS manager_index
        FROM similarity_edges
        UNION
        SELECT report_period, manager_b_index AS manager_index
        FROM similarity_edges
    ),
    edge_summary AS (
        SELECT
            report_period,
            COUNT(*) AS undirected_edges,
            COUNT(*) FILTER (WHERE is_mutual_neighbor) AS mutual_edges,
            AVG(is_mutual_neighbor::INTEGER) AS mutual_edge_share,
            AVG(cosine_similarity) AS mean_similarity,
            MEDIAN(cosine_similarity) AS median_similarity
        FROM similarity_edges
        GROUP BY report_period
    ),
    manager_summary AS (
        SELECT
            n.report_period,
            COUNT(*) AS managers,
            COUNT(e.manager_index) AS connected_managers
        FROM manager_nodes AS n
        LEFT JOIN endpoints AS e USING (report_period, manager_index)
        GROUP BY n.report_period
    ),
    calculated AS (
        SELECT
            e.*,
            m.connected_managers,
            m.managers - m.connected_managers AS isolated_managers,
            2.0 * e.undirected_edges
                / (m.managers * (m.managers - 1)) AS projected_density
        FROM edge_summary AS e
        INNER JOIN manager_summary AS m USING (report_period)
    )
    SELECT
        COUNT(*) FILTER (WHERE c.report_period IS NULL)
            AS audits_without_edges,
        COUNT(*) FILTER (WHERE a.report_period IS NULL)
            AS edges_without_audits,
        COUNT(*) FILTER (
            WHERE a.undirected_edges != c.undirected_edges
               OR a.mutual_edges != c.mutual_edges
               OR a.connected_managers != c.connected_managers
               OR a.isolated_managers != c.isolated_managers
               OR abs(a.mutual_edge_share - c.mutual_edge_share) > 1e-12
               OR abs(a.mean_similarity - c.mean_similarity) > 1e-12
               OR abs(a.median_similarity - c.median_similarity) > 1e-12
               OR abs(a.projected_density - c.projected_density) > 1e-12
        ) AS baseline_statistic_mismatches
    FROM saved_audit AS a
    FULL JOIN calculated AS c USING (report_period)
    WHERE a.neighbors_per_manager = 25 OR a.neighbors_per_manager IS NULL
    '''
).fetchdf().T.reset_index()
baseline_reconciliation.columns = ["check", "violations"]

display(baseline_reconciliation.style.format({"violations": "{:,}"}))
print()
print(
    "Total baseline-reconciliation violations: "
    f"{baseline_reconciliation['violations'].sum():,}"
)

,check,violations
0,audits_without_edges,0
1,edges_without_audits,0
2,baseline_statistic_mismatches,0



Total baseline-reconciliation violations: 0


## 4. Neighbor-count sensitivity

The 25-neighbor graph is the fixed baseline. The 10- and 50-neighbor alternatives show whether basic graph coverage and similarity patterns depend strongly on that choice.

In [4]:
similarity_audit = similarity_audit.sort_values(
    ["report_period", "neighbors_per_manager"]
)

audit_checks = pd.DataFrame(
    {
        "check": [
            "Duplicate quarter-neighbor rows",
            "Quarters without 10, 25, and 50 neighbors",
            "Rows with invalid audit statistics",
            "Quarters with decreasing edge counts",
        ],
        "violations": [
            similarity_audit.duplicated(
                ["report_period", "neighbors_per_manager"]
            ).sum(),
            similarity_audit.groupby("report_period")["neighbors_per_manager"]
                .apply(lambda values: set(values) != {10, 25, 50}).sum(),
            (
                (similarity_audit["undirected_edges"] <= 0)
                | (similarity_audit["mutual_edges"] < 0)
                | ~similarity_audit["mutual_edge_share"].between(0, 1)
                | ~similarity_audit["mean_similarity"].between(0, 1)
                | ~similarity_audit["median_similarity"].between(0, 1)
                | ~similarity_audit["projected_density"].between(0, 1)
                | (similarity_audit["connected_managers"] < 0)
                | (similarity_audit["isolated_managers"] < 0)
            ).sum(),
            similarity_audit.groupby("report_period")["undirected_edges"]
                .apply(lambda values: not values.is_monotonic_increasing).sum(),
        ],
    }
)

sensitivity_summary = (
    similarity_audit.groupby("neighbors_per_manager", as_index=False)
    .agg(
        median_edges=("undirected_edges", "median"),
        median_mutual_share=("mutual_edge_share", "median"),
        maximum_isolated_managers=("isolated_managers", "max"),
        median_similarity=("median_similarity", "median"),
        median_density=("projected_density", "median"),
    )
)

display(audit_checks.style.format({"violations": "{:,}"}))
display(sensitivity_summary.style.format({
    "median_edges": "{:,.0f}",
    "median_mutual_share": "{:.3f}",
    "maximum_isolated_managers": "{:,.0f}",
    "median_similarity": "{:.3f}",
    "median_density": "{:.4f}",
}))

connection.close()

,check,violations
0,Duplicate quarter-neighbor rows,0
1,"Quarters without 10, 25, and 50 neighbors",0
2,Rows with invalid audit statistics,0
3,Quarters with decreasing edge counts,0


,neighbors_per_manager,median_edges,median_mutual_share,maximum_isolated_managers,median_similarity,median_density
0,10,"40,368",0.125,0,0.567,0.0039
1,25,"101,550",0.119,0,0.523,0.0099
2,50,"204,997",0.107,0,0.486,0.0199


## 5. Neighbor-count sensitivity over time

The figure compares the quarterly similarity and density implied by 10, 25, and 50 neighbors. The Matplotlib and plotnine versions show the same data.

## 6. Interpretation

- **The production edge table is internally valid.** All 5,627,437 edges have unique ordered endpoints, valid positive cosine similarities, correct manager identifiers, and the fixed baseline value of 25 neighbors. Every edge check returns zero violations.
- **The baseline audit is reproducible.** Quarterly edge counts, mutual-link counts, connected managers, similarity statistics, and projected density agree exactly with the 25-neighbor edge table.
- **Every manager is connected.** The maximum number of isolated managers is zero under all three neighbor counts, so reducing the graph to 10 neighbors does not remove managers from the projected network.
- **The sensitivity pattern is mechanically coherent.** Raising the neighbor count increases median edge count and density, while median cosine similarity declines from 0.567 at 10 neighbors to 0.486 at 50 neighbors because progressively less similar portfolios are included. The 25-neighbor baseline lies between these alternatives, with median similarity 0.523.
- **The ordering is stable through time.** In every quarter, median similarity is highest for 10 neighbors and lowest for 50, while projected density follows the opposite ordering. The baseline therefore remains between the two sensitivity cases throughout the sample.

The saved similarity graphs are internally consistent and suitable for quarterly community detection. Similarity represents portfolio overlap, not a causal relationship or a permanent manager classification.